# M6.4 — Controlled training-window comparison

Milestone:

`M6 — Evaluation + Error Analysis`

Substep:

`M6.4 — Controlled Training-window Comparison`

Work type:

`RUNTIME COMPARATIVE EVALUATION`

Câu hỏi trung tâm:

> Trong từng model family, khi giữ model/config/features/preprocessing/validation/imbalance/threshold policy cố định và chỉ thay classifier-training window, W_SHORT và W_LONG tạo validation behavior khác nhau như thế nào?

Official controlled pairs:

```text
Logistic Regression:
M5-LR-SHORT-B04
vs
M5-LR-LONG-B04

Decision Tree:
M5-DT-SHORT-B01
vs
M5-DT-LONG-B01

Random Forest:
M5-RF-SHORT-B01
vs
M5-RF-LONG-B01
```

M6.4 không:

- fit/retrain model;
- tuning;
- imbalance intervention;
- threshold optimization;
- cross-model family selection;
- FINAL TEST access;
- khóa final training-window winner.

Runtime-dependent status trước khi chạy:

`NOT YET VERIFIED`

## 1. Contract kế thừa

M6.4 kế thừa trực tiếp:

- `CANON-M3.6 — Thiết kế protocol so sánh training window`;
- `CANON-Kế hoạch Milestone 6 — Evaluation và Error Analysis`;
- `CANON-M6.1 — Evaluation Charter / scope / guardrails`;
- `M6.2 — verified Evaluation Registry`;
- `M6.3 — verified six-run metric/confusion profiles`.

M3.6 khóa nguyên tắc:

```text
ONLY TRAINING WINDOW CHANGES
```

Validation:

```text
2019-01 → 2019-05
```

Primary comparison metric:

`F1_fraud`

Mandatory secondary evidence:

```text
Recall_fraud
Precision_fraud
TP
FP
FN
TN
predicted-positive count/rate
Accuracy — reference only
```

FINAL TEST:

`PROHIBITED FOR WINDOW SELECTION`

## 2. Interpretation boundary

M3.6 cho phép F1 tạo:

`PROVISIONAL LEADER`

nhưng không cho phép chỉ sort F1 rồi viết:

`W_SHORT/W_LONG thắng`

mà không đọc trade-off.

M6.4 sẽ:

1. kiểm tra controlled-pair identity;
2. tính delta;
3. xác định provisional F1 leader cho từng family;
4. đọc Recall / Precision;
5. đọc TP / FP / FN / TN;
6. đọc alert volume;
7. đọc computational context;
8. ghi pair-specific findings.

M6.4 **không** khóa global final training-window winner.

Final state phải giữ:

```text
Training-window winner:
OPEN

Model-family winner:
OPEN

Final model:
OPEN

Final threshold:
OPEN
```

Nếu cần robustness/tie-break:

`HANDOFF M7`

## 3. Delta convention

Mọi delta trong notebook được định nghĩa:

```text
DELTA = W_LONG - W_SHORT
```

Do đó:

```text
ΔF1 < 0
→ W_LONG thấp hơn W_SHORT về F1

ΔRecall < 0
→ W_LONG bắt ít fraud hơn W_SHORT

ΔFP > 0
→ W_LONG tạo nhiều false alert hơn W_SHORT

ΔFN > 0
→ W_LONG bỏ sót nhiều fraud hơn W_SHORT
```

Delta convention phải giữ nhất quán trong toàn M6.4.

In [1]:

from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np


print("Python:")
print(sys.version)

print("\nExecutable:")
print(sys.executable)

print("\nPlatform:")
print(platform.platform())

print("\nNumPy:")
print(np.__version__)


Python:
3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 21.0.0 (clang-2100.0.123.102)]

Executable:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/.venv/bin/python

Platform:
macOS-26.6.2-arm64-arm-64bit-Mach-O

NumPy:
2.5.3


## 4. Locate M6.2 / M6.3 artifacts

In [2]:

M6_02_REL = (
    Path("data")
    / "processed"
    / "m6_02_evaluation_artifact_audit"
)

M6_03_REL = (
    Path("data")
    / "processed"
    / "m6_03_baseline_metric_confusion_analysis"
)

M6_04_REL = (
    Path("data")
    / "processed"
    / "m6_04_training_window_comparison"
)


candidate_roots = [
    Path.cwd(),
    *list(Path.cwd().parents)[:6],
]


PROJECT_ROOT = None

required_rel_paths = [
    M6_02_REL
    / "m6_02_evaluation_registry.json",

    M6_03_REL
    / "m6_03_metric_confusion_analysis.json",

    M6_03_REL
    / "m6_03_analysis_manifest.json",
]


for candidate in candidate_roots:
    candidate = candidate.resolve()

    if all(
        (
            candidate
            / rel_path
        ).exists()
        for rel_path
        in required_rel_paths
    ):
        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy PROJECT_ROOT chứa "
        "M6.2 registry và M6.3 analysis artifacts."
    )


M6_02_DIR = (
    PROJECT_ROOT
    / M6_02_REL
)

M6_03_DIR = (
    PROJECT_ROOT
    / M6_03_REL
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / M6_04_REL
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


M6_02_REGISTRY_PATH = (
    M6_02_DIR
    / "m6_02_evaluation_registry.json"
)

M6_03_ANALYSIS_PATH = (
    M6_03_DIR
    / "m6_03_metric_confusion_analysis.json"
)

M6_03_MANIFEST_PATH = (
    M6_03_DIR
    / "m6_03_analysis_manifest.json"
)


print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nM6_02_REGISTRY_PATH:")
print(M6_02_REGISTRY_PATH)

print("\nM6_03_ANALYSIS_PATH:")
print(M6_03_ANALYSIS_PATH)

print("\nOUTPUT_DIR:")
print(OUTPUT_DIR)


PROJECT_ROOT:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening

M6_02_REGISTRY_PATH:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m6_02_evaluation_artifact_audit/m6_02_evaluation_registry.json

M6_03_ANALYSIS_PATH:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m6_03_baseline_metric_confusion_analysis/m6_03_metric_confusion_analysis.json

OUTPUT_DIR:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m6_04_training_window_comparison


## 5. Load và verify M6.3 handoff

In [3]:

with open(
    M6_02_REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as file:
    m6_02_registry = json.load(
        file
    )


with open(
    M6_03_ANALYSIS_PATH,
    "r",
    encoding="utf-8",
) as file:
    m6_03_analysis = json.load(
        file
    )


with open(
    M6_03_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    m6_03_manifest = json.load(
        file
    )


EXPECTED_M6_03_VERSION = (
    "M6.3-metric-confusion-analysis-v1"
)

EXPECTED_VALIDATION_ROWS = 712_458
EXPECTED_VALIDATION_FRAUD = 1_052


assert (
    m6_03_analysis[
        "analysis_version"
    ]
    == EXPECTED_M6_03_VERSION
)

assert (
    m6_03_analysis[
        "validation_rows"
    ]
    == EXPECTED_VALIDATION_ROWS
)

assert (
    m6_03_analysis[
        "validation_fraud_rows"
    ]
    == EXPECTED_VALIDATION_FRAUD
)

assert (
    len(
        m6_03_analysis[
            "records"
        ]
    )
    == 6
)

assert (
    m6_03_analysis[
        "selection_state"
    ][
        "training_window_winner"
    ]
    == "OPEN"
)

assert (
    m6_03_analysis[
        "selection_state"
    ][
        "model_family_winner"
    ]
    == "OPEN"
)

assert (
    m6_03_analysis[
        "selection_state"
    ][
        "m6_03_selection_authorized"
    ]
    is False
)

assert (
    m6_03_analysis[
        "final_test_accessed"
    ]
    is False
)


assert (
    m6_03_manifest[
        "official_runs_profiled"
    ]
    == 6
)

assert (
    m6_03_manifest[
        "confusion_arithmetic"
    ]
    == "PASS"
)

assert (
    m6_03_manifest[
        "metric_to_count_consistency"
    ]
    == "PASS"
)

assert (
    m6_03_manifest[
        "training_window_selection"
    ]
    == "NOT_PERFORMED"
)

assert (
    m6_03_manifest[
        "final_test_accessed"
    ]
    is False
)


print(
    "M6.4 M6.3 HANDOFF GATE: PASS"
)


M6.4 M6.3 HANDOFF GATE: PASS


## 6. Controlled pair definitions

Comparison IDs:

```text
TW-LR-B04
TW-DT-B01
TW-RF-B01
```

Mỗi pair phải có:

- cùng model family;
- cùng exact model config;
- cùng feature version;
- cùng preprocessing version;
- cùng matrix schema;
- cùng validation population;
- cùng imbalance strategy;
- cùng threshold policy;
- cùng risk-score semantics;
- khác `training_window_id`.

In [4]:

PAIR_SPECS = {
    "TW-LR-B04": {
        "family":
            "Logistic Regression",

        "short_id":
            "M5-LR-SHORT-B04",

        "long_id":
            "M5-LR-LONG-B04",

        "config_id":
            "LR-B04-LBFGS-L2-C1",
    },

    "TW-DT-B01": {
        "family":
            "Decision Tree",

        "short_id":
            "M5-DT-SHORT-B01",

        "long_id":
            "M5-DT-LONG-B01",

        "config_id":
            "DT-B01-DEFAULT-GINI-UNPRUNED",
    },

    "TW-RF-B01": {
        "family":
            "Random Forest",

        "short_id":
            "M5-RF-SHORT-B01",

        "long_id":
            "M5-RF-LONG-B01",

        "config_id":
            "RF-B01-100-GINI-SQRT-BOOTSTRAP",
    },
}


assert len(
    PAIR_SPECS
) == 3


profiles_by_id = {
    record[
        "experiment_id"
    ]:
    record
    for record
    in m6_03_analysis[
        "records"
    ]
}


registry_by_id = {
    record[
        "experiment_id"
    ]:
    record
    for record
    in m6_02_registry[
        "records"
    ]
}


for pair_id, spec in (
    PAIR_SPECS.items()
):
    assert (
        spec["short_id"]
        in profiles_by_id
    )

    assert (
        spec["long_id"]
        in profiles_by_id
    )

    assert (
        spec["short_id"]
        in registry_by_id
    )

    assert (
        spec["long_id"]
        in registry_by_id
    )


print(
    "Controlled pair specs:",
    len(
        PAIR_SPECS
    ),
)

print(
    "\nM6.4 PAIR DEFINITION GATE: PASS"
)


Controlled pair specs: 3

M6.4 PAIR DEFINITION GATE: PASS


## 7. Controlled-pair integrity audit

In [5]:

CONTROLLED_FIELDS = [
    "model_family",
    "model_id",
    "model_config_id",
    "feature_version",
    "preprocessing_version",
    "matrix_schema_version",
    "validation_period",
    "validation_rows",
    "validation_fraud_rows",
    "imbalance_strategy",
    "threshold_policy",
    "risk_score_kind",
]


pair_integrity_records = []


for pair_id, spec in (
    PAIR_SPECS.items()
):
    short_record = (
        registry_by_id[
            spec["short_id"]
        ]
    )

    long_record = (
        registry_by_id[
            spec["long_id"]
        ]
    )

    assert (
        short_record[
            "model_family"
        ]
        == spec["family"]
    )

    assert (
        long_record[
            "model_family"
        ]
        == spec["family"]
    )

    assert (
        short_record[
            "model_config_id"
        ]
        == spec["config_id"]
    )

    assert (
        long_record[
            "model_config_id"
        ]
        == spec["config_id"]
    )

    assert (
        short_record[
            "training_window_id"
        ]
        == "W_SHORT"
    )

    assert (
        long_record[
            "training_window_id"
        ]
        == "W_LONG"
    )

    mismatches = []

    for field in (
        CONTROLLED_FIELDS
    ):
        if (
            short_record[field]
            != long_record[field]
        ):
            mismatches.append(
                field
            )

    assert not mismatches, (
        f"{pair_id}: "
        f"controlled-field mismatch "
        f"{mismatches}"
    )

    pair_integrity_records.append(
        {
            "comparison_id":
                pair_id,

            "model_family":
                spec["family"],

            "controlled_fields_checked":
                len(
                    CONTROLLED_FIELDS
                ),

            "controlled_fields_match":
                True,

            "intended_difference":
                "training_window_id",

            "integrity_status":
                "PASS",
        }
    )


print(
    "Controlled fields checked per pair:",
    len(
        CONTROLLED_FIELDS
    ),
)

for record in (
    pair_integrity_records
):
    print(
        record[
            "comparison_id"
        ],
        "→",
        record[
            "integrity_status"
        ],
    )


print(
    "\nM6.4 CONTROLLED-PAIR INTEGRITY GATE: PASS"
)


Controlled fields checked per pair: 12
TW-LR-B04 → PASS
TW-DT-B01 → PASS
TW-RF-B01 → PASS

M6.4 CONTROLLED-PAIR INTEGRITY GATE: PASS


## 8. Load computational context từ persisted M5 summaries

M3.6 yêu cầu đọc computational cost cùng predictive evidence.

M6.4 không train lại model.

Notebook chỉ đọc:

```text
fit_seconds
prediction_seconds
warning_count
```

từ summary artifact đã được M6.2 identity-audit.

In [6]:

runtime_context = {}


for experiment_id, record in (
    registry_by_id.items()
):
    summary_path = (
        PROJECT_ROOT
        / record[
            "summary_artifact"
        ]
    )

    assert summary_path.exists()

    with open(
        summary_path,
        "r",
        encoding="utf-8",
    ) as file:
        summary = json.load(
            file
        )

    assert (
        summary[
            "experiment_id"
        ]
        == experiment_id
    )

    assert (
        summary[
            "final_test_accessed"
        ]
        is False
    )

    runtime_context[
        experiment_id
    ] = {
        "fit_seconds":
            float(
                summary[
                    "fit_seconds"
                ]
            ),

        "prediction_seconds":
            float(
                summary[
                    "prediction_seconds"
                ]
            ),

        "warning_count":
            int(
                summary[
                    "warning_count"
                ]
            ),
    }


assert len(
    runtime_context
) == 6


print(
    "M6.4 COMPUTATIONAL CONTEXT LOAD GATE: PASS"
)


M6.4 COMPUTATIONAL CONTEXT LOAD GATE: PASS


## 9. Build controlled comparison records

Delta convention:

`W_LONG − W_SHORT`

Primary decision evidence:

`F1_fraud`

Provisional leader:

- `W_SHORT` nếu SHORT F1 > LONG F1;
- `W_LONG` nếu LONG F1 > SHORT F1;
- `TIE_EXACT` chỉ nếu numerical equality exact trong stored evidence.

Không dùng arbitrary epsilon.

In [7]:

def build_pair_comparison(
    pair_id,
    spec,
):
    short_profile = (
        profiles_by_id[
            spec["short_id"]
        ]
    )

    long_profile = (
        profiles_by_id[
            spec["long_id"]
        ]
    )

    short_runtime = (
        runtime_context[
            spec["short_id"]
        ]
    )

    long_runtime = (
        runtime_context[
            spec["long_id"]
        ]
    )

    delta_fields = [
        "f1_fraud",
        "recall_fraud",
        "precision_fraud",
        "accuracy_reference",
        "tp",
        "fp",
        "fn",
        "tn",
        "alert_count",
        "alert_rate",
        "fraud_capture_count",
        "fraud_miss_count",
        "total_misclassified",
    ]

    deltas = {}

    for field in delta_fields:
        deltas[
            f"delta_long_minus_short__{field}"
        ] = (
            long_profile[field]
            - short_profile[field]
        )

    if (
        short_profile[
            "f1_fraud"
        ]
        > long_profile[
            "f1_fraud"
        ]
    ):
        provisional_f1_leader = (
            "W_SHORT"
        )
    elif (
        long_profile[
            "f1_fraud"
        ]
        > short_profile[
            "f1_fraud"
        ]
    ):
        provisional_f1_leader = (
            "W_LONG"
        )
    else:
        provisional_f1_leader = (
            "TIE_EXACT"
        )

    recall_leader = (
        "W_SHORT"
        if (
            short_profile[
                "recall_fraud"
            ]
            > long_profile[
                "recall_fraud"
            ]
        )
        else (
            "W_LONG"
            if (
                long_profile[
                    "recall_fraud"
                ]
                > short_profile[
                    "recall_fraud"
                ]
            )
            else "TIE_EXACT"
        )
    )

    precision_leader = (
        "W_SHORT"
        if (
            short_profile[
                "precision_fraud"
            ]
            > long_profile[
                "precision_fraud"
            ]
        )
        else (
            "W_LONG"
            if (
                long_profile[
                    "precision_fraud"
                ]
                > short_profile[
                    "precision_fraud"
                ]
            )
            else "TIE_EXACT"
        )
    )

    fit_ratio = (
        long_runtime[
            "fit_seconds"
        ]
        / short_runtime[
            "fit_seconds"
        ]
        if (
            short_runtime[
                "fit_seconds"
            ]
            > 0
        )
        else None
    )

    comparison = {
        "comparison_id":
            pair_id,

        "model_family":
            spec["family"],

        "model_config_id":
            spec["config_id"],

        "delta_convention":
            "W_LONG_MINUS_W_SHORT",

        "short_experiment_id":
            spec["short_id"],

        "long_experiment_id":
            spec["long_id"],

        "short":
            short_profile,

        "long":
            long_profile,

        "deltas":
            deltas,

        "short_runtime":
            short_runtime,

        "long_runtime":
            long_runtime,

        "fit_time_ratio_long_over_short":
            float(
                fit_ratio
            ),

        "provisional_f1_leader":
            provisional_f1_leader,

        "recall_leader":
            recall_leader,

        "precision_leader":
            precision_leader,

        "pair_integrity":
            "PASS",

        "window_specific_tuning":
            False,

        "threshold_policy_changed":
            False,

        "final_test_accessed":
            False,

        "pair_interpretation_scope":
            (
                "CONTROLLED_WITHIN_MODEL_FAMILY_"
                "VALIDATION_COMPARISON"
            ),

        "final_training_window_winner":
            "OPEN",
    }

    return comparison


pair_comparisons = [
    build_pair_comparison(
        pair_id,
        spec,
    )
    for pair_id, spec
    in PAIR_SPECS.items()
]


assert len(
    pair_comparisons
) == 3


print(
    "Pair comparisons:",
    len(
        pair_comparisons
    ),
)

print(
    "\nM6.4 COMPARISON BUILD GATE: PASS"
)


Pair comparisons: 3

M6.4 COMPARISON BUILD GATE: PASS


## 10. Print pair evidence

Runtime output phải hiển thị cho từng family:

- SHORT metrics;
- LONG metrics;
- LONG − SHORT deltas;
- TP/FP/FN;
- alert volume;
- provisional F1 leader;
- Recall leader;
- Precision leader;
- fit-time context.

Không in global winner.

In [8]:

def print_pair_comparison(
    comparison,
):
    short = comparison[
        "short"
    ]

    long = comparison[
        "long"
    ]

    deltas = comparison[
        "deltas"
    ]

    print("=" * 72)

    print(
        "Comparison:",
        comparison[
            "comparison_id"
        ],
    )

    print(
        "Family:",
        comparison[
            "model_family"
        ],
    )

    print(
        "Config:",
        comparison[
            "model_config_id"
        ],
    )

    print()

    print("W_SHORT")

    print(
        "  F1:",
        short[
            "f1_fraud"
        ],
    )

    print(
        "  Recall:",
        short[
            "recall_fraud"
        ],
    )

    print(
        "  Precision:",
        short[
            "precision_fraud"
        ],
    )

    print(
        "  TP / FP / FN / TN:",
        short["tp"],
        short["fp"],
        short["fn"],
        short["tn"],
    )

    print(
        "  Alerts:",
        short[
            "alert_count"
        ],
    )

    print(
        "  Fit seconds:",
        comparison[
            "short_runtime"
        ][
            "fit_seconds"
        ],
    )

    print()

    print("W_LONG")

    print(
        "  F1:",
        long[
            "f1_fraud"
        ],
    )

    print(
        "  Recall:",
        long[
            "recall_fraud"
        ],
    )

    print(
        "  Precision:",
        long[
            "precision_fraud"
        ],
    )

    print(
        "  TP / FP / FN / TN:",
        long["tp"],
        long["fp"],
        long["fn"],
        long["tn"],
    )

    print(
        "  Alerts:",
        long[
            "alert_count"
        ],
    )

    print(
        "  Fit seconds:",
        comparison[
            "long_runtime"
        ][
            "fit_seconds"
        ],
    )

    print()

    print(
        "DELTA = W_LONG - W_SHORT"
    )

    print(
        "  ΔF1:",
        deltas[
            "delta_long_minus_short__f1_fraud"
        ],
    )

    print(
        "  ΔRecall:",
        deltas[
            "delta_long_minus_short__recall_fraud"
        ],
    )

    print(
        "  ΔPrecision:",
        deltas[
            "delta_long_minus_short__precision_fraud"
        ],
    )

    print(
        "  ΔTP:",
        deltas[
            "delta_long_minus_short__tp"
        ],
    )

    print(
        "  ΔFP:",
        deltas[
            "delta_long_minus_short__fp"
        ],
    )

    print(
        "  ΔFN:",
        deltas[
            "delta_long_minus_short__fn"
        ],
    )

    print(
        "  ΔAlerts:",
        deltas[
            "delta_long_minus_short__alert_count"
        ],
    )

    print(
        "  Fit ratio LONG / SHORT:",
        comparison[
            "fit_time_ratio_long_over_short"
        ],
    )

    print()

    print(
        "Provisional F1 leader:",
        comparison[
            "provisional_f1_leader"
        ],
    )

    print(
        "Recall leader:",
        comparison[
            "recall_leader"
        ],
    )

    print(
        "Precision leader:",
        comparison[
            "precision_leader"
        ],
    )

    print(
        "Final training-window winner:",
        comparison[
            "final_training_window_winner"
        ],
    )


for comparison in (
    pair_comparisons
):
    print_pair_comparison(
        comparison
    )


print(
    "\nM6.4 THREE CONTROLLED PAIR EVIDENCE: READY"
)


Comparison: TW-LR-B04
Family: Logistic Regression
Config: LR-B04-LBFGS-L2-C1

W_SHORT
  F1: 0.33747547416612167
  Recall: 0.24524714828897337
  Precision: 0.5408805031446541
  TP / FP / FN / TN: 258 219 794 711187
  Alerts: 477
  Fit seconds: 0.8222020840039477

W_LONG
  F1: 0.03996366939146231
  Recall: 0.02091254752851711
  Precision: 0.4489795918367347
  TP / FP / FN / TN: 22 27 1030 711379
  Alerts: 49
  Fit seconds: 3.6143539589829743

DELTA = W_LONG - W_SHORT
  ΔF1: -0.2975118047746594
  ΔRecall: -0.22433460076045625
  ΔPrecision: -0.09190091130791939
  ΔTP: -236
  ΔFP: -192
  ΔFN: 236
  ΔAlerts: -428
  Fit ratio LONG / SHORT: 4.395943563389971

Provisional F1 leader: W_SHORT
Recall leader: W_SHORT
Precision leader: W_SHORT
Final training-window winner: OPEN
Comparison: TW-DT-B01
Family: Decision Tree
Config: DT-B01-DEFAULT-GINI-UNPRUNED

W_SHORT
  F1: 0.3270564915758176
  Recall: 0.31368821292775667
  Precision: 0.3416149068322981
  TP / FP / FN / TN: 330 636 722 710770
  Alerts

## 11. Predictive-trade-off classification

M6.4 ghi pair behavior theo rule mô tả:

### `SHORT_DOMINATES_F1_RECALL_PRECISION`

W_SHORT cao hơn cả:

- F1;
- Recall;
- Precision.

### `SHORT_F1_RECALL__LONG_PRECISION`

W_SHORT cao hơn:

- F1;
- Recall;

nhưng W_LONG cao hơn:

- Precision.

### `LONG_DOMINATES_F1_RECALL_PRECISION`

W_LONG cao hơn cả ba.

### `MIXED_OTHER`

Các pattern khác.

Đây là descriptive classification, không phải final winner rule.

In [9]:

def classify_tradeoff(
    comparison,
):
    f1_leader = comparison[
        "provisional_f1_leader"
    ]

    recall_leader = comparison[
        "recall_leader"
    ]

    precision_leader = comparison[
        "precision_leader"
    ]

    if (
        f1_leader == "W_SHORT"
        and recall_leader == "W_SHORT"
        and precision_leader == "W_SHORT"
    ):
        return (
            "SHORT_DOMINATES_"
            "F1_RECALL_PRECISION"
        )

    if (
        f1_leader == "W_SHORT"
        and recall_leader == "W_SHORT"
        and precision_leader == "W_LONG"
    ):
        return (
            "SHORT_F1_RECALL__"
            "LONG_PRECISION"
        )

    if (
        f1_leader == "W_LONG"
        and recall_leader == "W_LONG"
        and precision_leader == "W_LONG"
    ):
        return (
            "LONG_DOMINATES_"
            "F1_RECALL_PRECISION"
        )

    return "MIXED_OTHER"


for comparison in (
    pair_comparisons
):
    comparison[
        "tradeoff_class"
    ] = classify_tradeoff(
        comparison
    )

    print(
        comparison[
            "comparison_id"
        ],
        "→",
        comparison[
            "tradeoff_class"
        ],
    )


print(
    "\nM6.4 TRADE-OFF CLASSIFICATION GATE: PASS"
)


TW-LR-B04 → SHORT_DOMINATES_F1_RECALL_PRECISION
TW-DT-B01 → SHORT_DOMINATES_F1_RECALL_PRECISION
TW-RF-B01 → SHORT_F1_RECALL__LONG_PRECISION

M6.4 TRADE-OFF CLASSIFICATION GATE: PASS


## 12. Cross-pair consistency diagnostic

M6.4 được phép hỏi:

> Provisional F1 leader có cùng hướng trong cả 3 controlled pairs không?

Đây **không** phải cross-model family selection.

Nó chỉ tổng hợp consistency của training-window direction.

Nếu cả ba pair cùng hướng:

ghi:

`CONSISTENT PROVISIONAL DIRECTION`

nhưng final training-window winner vẫn:

`OPEN`

cho đến khi protocol selection/robustness ở giai đoạn phù hợp cho phép khóa.

In [10]:

provisional_f1_leaders = [
    comparison[
        "provisional_f1_leader"
    ]
    for comparison
    in pair_comparisons
]


unique_provisional_f1_leaders = set(
    provisional_f1_leaders
)


if (
    len(
        unique_provisional_f1_leaders
    )
    == 1
):
    cross_pair_direction = (
        "CONSISTENT_PROVISIONAL_DIRECTION"
    )

    common_provisional_direction = (
        provisional_f1_leaders[0]
    )
else:
    cross_pair_direction = (
        "MIXED_PROVISIONAL_DIRECTION"
    )

    common_provisional_direction = (
        "NONE"
    )


print(
    "Provisional F1 leaders:",
    provisional_f1_leaders,
)

print(
    "Cross-pair direction:",
    cross_pair_direction,
)

print(
    "Common provisional direction:",
    common_provisional_direction,
)

print(
    "Final training-window winner:",
    "OPEN",
)


print(
    "\nM6.4 CROSS-PAIR DIRECTION DIAGNOSTIC: READY"
)


Provisional F1 leaders: ['W_SHORT', 'W_SHORT', 'W_SHORT']
Cross-pair direction: CONSISTENT_PROVISIONAL_DIRECTION
Common provisional direction: W_SHORT
Final training-window winner: OPEN

M6.4 CROSS-PAIR DIRECTION DIAGNOSTIC: READY


## 13. No arbitrary epsilon / no automatic winner

M3.6 cấm rule kiểu:

```text
|ΔF1| < 0.01
→ tie
```

M6.4 không tạo epsilon.

M6.4 cũng không tự động chuyển:

`provisional F1 leader`

thành:

`final training-window winner`.

Nếu comparative evidence cần robustness:

`M7 temporal robustness / CV`

theo upstream policy.

In [11]:

selection_state = {
    "training_window_winner":
        "OPEN",

    "model_family_winner":
        "OPEN",

    "final_model":
        "OPEN",

    "final_threshold":
        "OPEN",

    "arbitrary_f1_epsilon_used":
        False,

    "window_specific_tuning_used":
        False,

    "threshold_optimization_used":
        False,

    "m6_04_final_selection_authorized":
        False,
}


assert (
    selection_state[
        "training_window_winner"
    ]
    == "OPEN"
)

assert (
    selection_state[
        "arbitrary_f1_epsilon_used"
    ]
    is False
)

assert (
    selection_state[
        "window_specific_tuning_used"
    ]
    is False
)

assert (
    selection_state[
        "threshold_optimization_used"
    ]
    is False
)

assert (
    selection_state[
        "m6_04_final_selection_authorized"
    ]
    is False
)


print(
    "Training-window winner:",
    selection_state[
        "training_window_winner"
    ],
)

print(
    "Arbitrary F1 epsilon used:",
    selection_state[
        "arbitrary_f1_epsilon_used"
    ],
)

print(
    "\nM6.4 SELECTION-BOUNDARY GATE: PASS"
)


Training-window winner: OPEN
Arbitrary F1 epsilon used: False

M6.4 SELECTION-BOUNDARY GATE: PASS


## 14. Source-artifact fingerprints

In [12]:

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


m6_02_registry_sha256 = (
    sha256_file(
        M6_02_REGISTRY_PATH
    )
)

m6_03_analysis_sha256 = (
    sha256_file(
        M6_03_ANALYSIS_PATH
    )
)


print(
    "M6.2 registry SHA256:"
)
print(
    m6_02_registry_sha256
)

print(
    "\nM6.3 analysis SHA256:"
)
print(
    m6_03_analysis_sha256
)


M6.2 registry SHA256:
9e112883d5aa1358c58c714edc0f0ef7913252e94c56f3c6b28d3ba1b7f4e6fb

M6.3 analysis SHA256:
290f348ba20a4a19f659e24bb517ca8cce2229605af3f7ee762520e5b1d7b168


## 15. Persist M6.4 controlled-comparison artifact

In [13]:

COMPARISON_VERSION = (
    "M6.4-training-window-comparison-v1"
)

COMPARISON_PATH = (
    OUTPUT_DIR
    / "m6_04_training_window_comparison.json"
)

MANIFEST_PATH = (
    OUTPUT_DIR
    / "m6_04_comparison_manifest.json"
)


comparison_payload = {
    "comparison_version":
        COMPARISON_VERSION,

    "m6_substep":
        "M6.4",

    "delta_convention":
        "W_LONG_MINUS_W_SHORT",

    "source_m6_02_registry":
        str(
            M6_02_REGISTRY_PATH.relative_to(
                PROJECT_ROOT
            )
        ),

    "source_m6_02_registry_sha256":
        m6_02_registry_sha256,

    "source_m6_03_analysis":
        str(
            M6_03_ANALYSIS_PATH.relative_to(
                PROJECT_ROOT
            )
        ),

    "source_m6_03_analysis_sha256":
        m6_03_analysis_sha256,

    "validation_rows":
        EXPECTED_VALIDATION_ROWS,

    "validation_fraud_rows":
        EXPECTED_VALIDATION_FRAUD,

    "primary_metric":
        "F1_fraud",

    "mandatory_secondary_evidence": [
        "Recall_fraud",
        "Precision_fraud",
        "TP",
        "FP",
        "FN",
        "TN",
        "predicted_positive_count",
        "predicted_positive_rate",
        "Accuracy_reference",
    ],

    "comparison_pairs":
        pair_comparisons,

    "cross_pair_direction":
        cross_pair_direction,

    "common_provisional_direction":
        common_provisional_direction,

    "selection_state":
        selection_state,

    "final_test_accessed":
        False,
}


comparison_manifest = {
    "m6_substep":
        "M6.4",

    "comparison_version":
        COMPARISON_VERSION,

    "controlled_pairs_expected":
        3,

    "controlled_pairs_built":
        len(
            pair_comparisons
        ),

    "controlled_pair_integrity":
        "PASS",

    "delta_convention":
        "W_LONG_MINUS_W_SHORT",

    "primary_metric":
        "F1_fraud",

    "arbitrary_f1_epsilon_used":
        False,

    "window_specific_tuning":
        "NOT_PERFORMED",

    "threshold_optimization":
        "NOT_PERFORMED",

    "final_training_window_selected":
        False,

    "cross_model_family_selection":
        "NOT_PERFORMED",

    "final_test_accessed":
        False,

    "decision":
        "OPEN — REQUIRES M6.4 RUNTIME REVIEW",
}


with open(
    COMPARISON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        comparison_payload,
        file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        comparison_manifest,
        file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


assert COMPARISON_PATH.exists()
assert MANIFEST_PATH.exists()

assert COMPARISON_PATH.stat().st_size > 0
assert MANIFEST_PATH.stat().st_size > 0


print("Comparison artifact:")
print(COMPARISON_PATH)

print("\nComparison manifest:")
print(MANIFEST_PATH)

print(
    "\nM6.4 COMPARISON PERSISTENCE GATE: PASS"
)


Comparison artifact:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m6_04_training_window_comparison/m6_04_training_window_comparison.json

Comparison manifest:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m6_04_training_window_comparison/m6_04_comparison_manifest.json

M6.4 COMPARISON PERSISTENCE GATE: PASS


## 16. Persisted artifact round-trip

In [14]:

with open(
    COMPARISON_PATH,
    "r",
    encoding="utf-8",
) as file:
    comparison_roundtrip = json.load(
        file
    )


with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    manifest_roundtrip = json.load(
        file
    )


assert (
    comparison_roundtrip[
        "comparison_version"
    ]
    == COMPARISON_VERSION
)

assert (
    len(
        comparison_roundtrip[
            "comparison_pairs"
        ]
    )
    == 3
)

assert {
    pair[
        "comparison_id"
    ]
    for pair
    in comparison_roundtrip[
        "comparison_pairs"
    ]
} == set(
    PAIR_SPECS.keys()
)

assert (
    comparison_roundtrip[
        "selection_state"
    ][
        "training_window_winner"
    ]
    == "OPEN"
)

assert (
    comparison_roundtrip[
        "selection_state"
    ][
        "m6_04_final_selection_authorized"
    ]
    is False
)

assert (
    comparison_roundtrip[
        "final_test_accessed"
    ]
    is False
)

assert (
    manifest_roundtrip[
        "controlled_pairs_built"
    ]
    == 3
)

assert (
    manifest_roundtrip[
        "controlled_pair_integrity"
    ]
    == "PASS"
)

assert (
    manifest_roundtrip[
        "arbitrary_f1_epsilon_used"
    ]
    is False
)

assert (
    manifest_roundtrip[
        "final_training_window_selected"
    ]
    is False
)


print(
    "M6.4 COMPARISON ROUND-TRIP GATE: PASS"
)


M6.4 COMPARISON ROUND-TRIP GATE: PASS


## 17. FINAL TEST isolation

In [15]:

assert (
    m6_02_registry[
        "final_test_accessed"
    ]
    is False
)

assert (
    m6_03_analysis[
        "final_test_accessed"
    ]
    is False
)


for comparison in (
    pair_comparisons
):
    assert (
        comparison[
            "final_test_accessed"
        ]
        is False
    )


assert (
    comparison_payload[
        "final_test_accessed"
    ]
    is False
)

assert (
    comparison_manifest[
        "final_test_accessed"
    ]
    is False
)


for path in [
    M6_02_REGISTRY_PATH,
    M6_03_ANALYSIS_PATH,
    M6_03_MANIFEST_PATH,
]:
    assert (
        "final_test"
        not in path.name.lower()
    )


print(
    "M6.4 FINAL TEST ISOLATION GATE: PASS"
)


M6.4 FINAL TEST ISOLATION GATE: PASS


## 18. M6.4 overall technical gate

In [16]:

m6_04_gates = {
    "G01_M6_03_HANDOFF":
        True,

    "G02_PAIR_DEFINITION":
        True,

    "G03_CONTROLLED_PAIR_INTEGRITY":
        True,

    "G04_COMPUTATIONAL_CONTEXT":
        True,

    "G05_COMPARISON_BUILD":
        True,

    "G06_THREE_PAIR_EVIDENCE":
        True,

    "G07_TRADEOFF_CLASSIFICATION":
        True,

    "G08_CROSS_PAIR_DIRECTION":
        True,

    "G09_NO_ARBITRARY_EPSILON":
        True,

    "G10_SELECTION_BOUNDARY":
        True,

    "G11_SOURCE_LINEAGE":
        True,

    "G12_COMPARISON_PERSISTENCE":
        True,

    "G13_COMPARISON_ROUND_TRIP":
        True,

    "G14_FINAL_TEST_ISOLATION":
        True,
}


for gate_name, gate_value in (
    m6_04_gates.items()
):
    print(
        gate_name,
        "→",
        (
            "PASS"
            if gate_value
            else "FAIL"
        ),
    )


assert all(
    m6_04_gates.values()
)


print(
    "\nM6.4 OVERALL TECHNICAL GATE: PASS"
)


G01_M6_03_HANDOFF → PASS
G02_PAIR_DEFINITION → PASS
G03_CONTROLLED_PAIR_INTEGRITY → PASS
G04_COMPUTATIONAL_CONTEXT → PASS
G05_COMPARISON_BUILD → PASS
G06_THREE_PAIR_EVIDENCE → PASS
G07_TRADEOFF_CLASSIFICATION → PASS
G08_CROSS_PAIR_DIRECTION → PASS
G09_NO_ARBITRARY_EPSILON → PASS
G10_SELECTION_BOUNDARY → PASS
G11_SOURCE_LINEAGE → PASS
G12_COMPARISON_PERSISTENCE → PASS
G13_COMPARISON_ROUND_TRIP → PASS
G14_FINAL_TEST_ISOLATION → PASS

M6.4 OVERALL TECHNICAL GATE: PASS


# 19. Runtime review registry

Sau Run All, cần review các evidence sau.

## M6.4-F01 — Execution integrity

Cần xác minh:

- toàn bộ code cells đã chạy;
- không error;
- không stderr bất thường.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F02 — Controlled-pair integrity

Cần xác minh:

```text
LR pair:
PASS

DT pair:
PASS

RF pair:
PASS
```

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F03 — Logistic Regression window comparison

Cần diễn giải:

```text
W_SHORT vs W_LONG
ΔF1
ΔRecall
ΔPrecision
ΔTP
ΔFP
ΔFN
ΔAlerts
fit-time context
```

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F04 — Decision Tree window comparison

Cần diễn giải cùng canonical evidence bundle.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F05 — Random Forest window comparison

Cần diễn giải cùng canonical evidence bundle.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F06 — Provisional F1 leaders

Cần ghi pair-specific provisional leaders theo M3.6.

Không tự chuyển thành final winner.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F07 — Recall / Precision trade-off

Cần xác minh có pair nào:

- SHORT dẫn F1/Recall/Precision cùng lúc;
- hoặc SHORT dẫn F1/Recall nhưng LONG dẫn Precision.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F08 — Confusion-count deltas

Cần diễn giải delta về fraud caught/missed và false alerts.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F09 — Computational context

Cần đọc fit-time ratio nhưng không dùng runtime làm automatic winner rule.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F10 — Cross-pair direction

Cần xác minh provisional F1 direction có nhất quán qua 3 family không.

Final window vẫn OPEN.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F11 — Selection boundary

Cần xác minh:

```text
Training-window winner:
OPEN

Model-family winner:
OPEN

Final model:
OPEN

Final threshold:
OPEN
```

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F12 — FINAL TEST

Cần xác minh no access.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.4-F13 — M6.5 handoff

Chỉ READY sau runtime review.

Status:

`OPEN — PENDING RUNTIME REVIEW`

# 20. Decision Log — trước runtime review

## M6.4-D01 — Comparison target

Decision:

Compare classifier-training window.

Status:

`INHERITED — LOCKED`

---

## M6.4-D02 — Official window candidates

Decision:

```text
W_SHORT
W_LONG
```

Status:

`INHERITED — LOCKED`

---

## M6.4-D03 — Controlled-pair rule

Decision:

Within family:

`ONLY TRAINING WINDOW CHANGES`

Status:

`INHERITED — LOCKED`

---

## M6.4-D04 — Validation

Decision:

Same canonical validation:

```text
2019-01 → 2019-05
712,458 rows
1,052 fraud
```

Status:

`INHERITED — LOCKED`

---

## M6.4-D05 — Primary comparison metric

Decision:

`F1_fraud`

Status:

`INHERITED — LOCKED`

---

## M6.4-D06 — Mandatory secondary evidence

Decision:

```text
Recall
Precision
TP / FP / FN / TN
predicted-positive count/rate
Accuracy reference
```

Status:

`INHERITED — LOCKED`

---

## M6.4-D07 — Delta convention

Decision:

`W_LONG − W_SHORT`

Status:

`LOCKED FOR M6.4`

---

## M6.4-D08 — Provisional leader

Decision:

Higher F1 identifies pair-specific:

`PROVISIONAL F1 LEADER`

not final winner.

Status:

`LOCKED`

---

## M6.4-D09 — Arbitrary epsilon

Decision:

No arbitrary ΔF1 epsilon.

Status:

`INHERITED — LOCKED`

---

## M6.4-D10 — Computational context

Decision:

Use persisted fit/prediction runtime descriptively.

Do not use runtime as automatic predictive winner rule.

Status:

`LOCKED`

---

## M6.4-D11 — Window-specific tuning

Decision:

None.

Status:

`INHERITED — LOCKED`

---

## M6.4-D12 — Threshold

Decision:

Same baseline threshold policy.

No optimization.

Status:

`INHERITED — LOCKED`

---

## M6.4-D13 — Global training-window winner

Decision:

M6.4 does not lock final global winner.

Status:

`OPEN — HANDOFF M7 IF SELECTION/ROBUSTNESS REQUIRED`

---

## M6.4-D14 — Cross-model family selection

Decision:

Not performed in M6.4.

Status:

`DEFERRED TO M6.5 / M7`

---

## M6.4-D15 — Comparison artifact

Decision:

Persist:

`m6_04_training_window_comparison.json`

Status:

`LOCKED`

---

## M6.4-D16 — FINAL TEST

Decision:

No access.

Status:

`INHERITED — LOCKED`

---

## M6.4-D17 — M6.5 handoff

Status:

`OPEN — REQUIRES RUNTIME REVIEW`

# 21. Kết luận trước runtime

Notebook M6.4 đã được thiết kế để:

- consume verified M6.2/M6.3 evidence;
- verify 3 controlled pairs;
- compute `W_LONG − W_SHORT` deltas;
- read F1 first;
- read Recall / Precision trade-off;
- translate differences into TP/FP/FN/TN;
- read alert-volume changes;
- read computational context;
- identify pair-specific provisional F1 leaders;
- summarize cross-pair direction;
- persist controlled-comparison evidence;
- preserve selection boundary;
- protect FINAL TEST.

Trước Run All:

```text
M6.4:
NOT YET VERIFIED

Controlled pairs:
3

Delta convention:
W_LONG - W_SHORT

Primary metric:
F1_fraud

Secondary:
Recall_fraud
Precision_fraud

Mandatory:
TP / FP / FN / TN
alert count/rate

Provisional F1 leader:
TO BE OBSERVED

Global training-window winner:
OPEN

Cross-model selection:
NOT PERFORMED

Final model:
OPEN

Final threshold:
OPEN

FINAL TEST:
PROTECTED

Next:
Run All → runtime review
```

Chỉ sau runtime review mới được khóa:

`M6.4 — PASS`

và:

`READY FOR M6.5`